In [ ]:
from collections import Counter, defaultdict
import numbers
from pathlib import Path

import fitz
import numpy as np
import pandas as pd

pdf_base_dir = "path/to/PDFs Literaturdatenbank"
dev_set_coreschema_pdf_dir = f"{pdf_base_dir}/dev-set-100"
dev_set_organismtrend_pdf_dir = f"{pdf_base_dir}/dev-set-Wald-WVC"
test_set_coreschema_pdf_dir = f"{pdf_base_dir}/test"
test_set_organismtrend_pdf_dir = f"{pdf_base_dir}/test-set-AuO-WVC"
dev_set_coreschema_references = "../data/interim/faktencheck-db/faktenscheck_core_corrected.jsonl"
dev_set_coreschema_prediction_texts = "../data/processed/faktencheck/dev-set-100/predictions.jsonl"
dev_set_organismtrend_references = (
    "../data/external/organism_trends/Referenz_Wald_korrigiert_angepasst.csv"
)
dev_set_organismtrend_prediction_texts = (
    "../data/processed/faktencheck/dev-set-Wald-WVC/predictions.jsonl.gz"
)
test_set_coreschema_references = (
    "../data/interim/faktencheck-db/faktencheck-db-converted_2025-11-05.jsonl"
)
test_set_coreschema_prediction_texts = "../data/processed/faktencheck/test/predictions.jsonl.gz"
test_set_organismtrend_references = "../data/external/organism_trends/Weighted Vote Count Agrar- und Offenland Literatur - Sheet1.csv"
test_set_organismtrend_prediction_texts = (
    "../data/processed/faktencheck/test-set-AuO-WVC/predictions.jsonl.gz"
)

In [ ]:
def count_column_values(col_name, value, val_counters, val_totals, val_sums):
    """Flattens lists/dicts and updates counters."""
    if value is None:
        return

    # dict → rekursiv flatten
    if isinstance(value, dict):
        for k, v in value.items():
            count_column_values(f"{col_name}.{k}", v, val_counters, val_totals, val_sums)
        return

    # list → jedes Element einzeln
    if isinstance(value, list):
        for item in value:
            count_column_values(col_name, item, val_counters, val_totals, val_sums)
        return

    # primitive value
    val_counters[col_name][value] += 1
    val_totals[col_name] += 1

    if isinstance(value, numbers.Number):
        val_sums[col_name] += value


def count_pages(df, col, base_dir):
    page_counts = []

    for file_path in df[col].dropna().unique():
        try:
            doc = fitz.open(Path(base_dir, file_path))
            num_pages = doc.page_count
            page_counts.append(num_pages)
            doc.close()

        except Exception as e:
            print(f"Error reading {file_path}: {e}")

    sum_pages = np.sum(page_counts) if page_counts else 0
    avg_pages = np.mean(page_counts) if page_counts else 0
    med_pages = np.median(page_counts) if page_counts else 0
    max_pages = max(page_counts) if page_counts else 0
    min_pages = min(page_counts) if page_counts else 0

    print(f"\n\nColumn: {col}")
    print(
        f"Total PDF files (in dataframe / with openable PDF): {len(df[col].dropna().unique())} / {len(page_counts)}"
    )
    print(f"Average pages per PDF: {avg_pages:.2f}")
    print(f"Median pages per PDF: {med_pages}")
    print(f"Max pages: {max_pages}")
    print(f"Min pages: {min_pages}")
    print(f"Total pages: {sum_pages}")


def count_words(df, col):
    word_counts = []

    for value in df[col].dropna():
        if not isinstance(value, str):
            continue

        words = value.split()  # whitespace split
        word_counts.append(len(words))

    # Summary stats
    sum_words = np.sum(word_counts) if word_counts else 0
    avg_words = np.mean(word_counts) if word_counts else 0
    med_words = np.median(word_counts) if word_counts else 0
    max_words = max(word_counts) if word_counts else 0
    min_words = min(word_counts) if word_counts else 0

    print(f"\n\nColumn: {col}")
    print(f"Average words per doc: {avg_words:.2f}")
    print(f"Median words per doc: {med_words}")
    print(f"Max words per doc: {max_words}")
    print(f"Min words per doc: {min_words}")
    print(f"Total words: {sum_words}")


def count_df_values(df, show_most_common=True):
    val_counters = defaultdict(Counter)
    val_totals = defaultdict(int)
    val_sums = defaultdict(float)

    for col in df.columns:
        for value in df[col]:
            count_column_values(col, value, val_counters, val_totals, val_sums)

    # Output
    for col in sorted(val_counters):
        print(f"\n=== {col} ===")
        print(f"Total values: {val_totals[col]}")

        if val_sums[col] != 0:
            print(f"Numeric sum: {val_sums[col]}")

        if show_most_common:
            for val, count in val_counters[col].most_common():
                print(f"  {val}: {count}")

In [ ]:
# Dev set Core Schema reference stats
df = pd.read_json(dev_set_coreschema_references, lines=True)
print(df.head())

print("\n=== Unique gold entries (support) in the Dev-set CoreSchema reference data ===")
print(f"Unique docs with annotations: {df.notna().any(axis=1).sum()}")
count_df_values(
    df[["biodiversity_level", "ecosystem_type", "habitat", "taxa"]], show_most_common=False
)

In [ ]:
# Dev set Core schema text stats
df = pd.read_json(dev_set_coreschema_prediction_texts, lines=True)
print(df.head())

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, dev_set_coreschema_pdf_dir)

In [ ]:
# Dev set OrganismTrend Schema reference stats
df = pd.read_csv(dev_set_organismtrend_references)
df = df[df["Key"] != "#NV"]
print(df.head())

print(
    "\n=== Unique gold entries (support) in the OrganismTrend dev-set-Wald-WVC reference data ==="
)
print(f"Unique docs with annotations: {len(df['Key'].unique())}")

count_df_values(df[["Antwortvariable", "Lebensraum", "Trend", "Hauptgruppe_RoteListen"]])

In [ ]:
# Dev set OrganismTrend schema text stats
df = pd.read_json(dev_set_organismtrend_prediction_texts, lines=True)
print(df.head())

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, dev_set_organismtrend_pdf_dir)

In [ ]:
# Test set Core Schema reference stats
df = pd.read_json(test_set_coreschema_references, lines=True)
# filter down to test set PDF ids
valid_test_set_docs = pd.read_json(test_set_coreschema_prediction_texts, lines=True)
df = df[df["zotitem_ptr_id"].isin(valid_test_set_docs["file_name"].str.removesuffix(".pdf"))]
print(df.head())

print("\n=== Unique gold entries (support) in the Test-set CoreSchema reference data ===")
print(f"Unique docs with annotations: {df.notna().any(axis=1).sum()}")
count_df_values(
    df[["biodiversity_level", "ecosystem_type", "habitat", "taxa"]], show_most_common=False
)

In [ ]:
# Test set Core schema text stats
df = pd.read_json(test_set_coreschema_prediction_texts, lines=True)
print(df.head())

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, test_set_coreschema_pdf_dir)

In [ ]:
# Test set OrganismTrend Schema reference stats
df = pd.read_csv(test_set_organismtrend_references)
df = df[df["Key"] != "#NV"]
print(df.head())

print(
    "\n=== Unique gold entries (support) in the OrganismTrend test-set-AuO-WVC reference data ==="
)
print(f"Unique docs with annotations: {len(df['Key'].unique())}")

count_df_values(df[["Antwortvariable", "Lebensraum", "Trend", "Hauptgruppe_RoteListen"]])

In [ ]:
# Test set OrganismTrend schema text stats
df = pd.read_json(test_set_organismtrend_prediction_texts, lines=True)
print(df.head())

col = "text"
count_words(df, col)

col = "file_name"
count_pages(df, col, test_set_organismtrend_pdf_dir)